## 1️⃣ Environment Setup & GPU Verification

In [23]:
# ============================================================
# 1.1 Verify GPU is available
# ============================================================
import subprocess

print("=" * 70)
print("🔍 Checking GPU availability...")
print("=" * 70)

# Check NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print("❌ ERROR: No NVIDIA GPU detected!")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
    raise RuntimeError("GPU not available")

print("\n✅ GPU detected! Proceeding with setup...")

🔍 Checking GPU availability...
Tue Jan  6 16:46:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             61W /  400W |    1469MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------

In [24]:
# ============================================================
# 1.2 Install dependencies (Colab with TF 2.19.0)
# ============================================================
# ⚠️ If you see numpy/h5py errors, go to Runtime > Restart runtime
# then run cells 1 and 2 again (skip pip install, just verify)

print("📦 Checking/Installing dependencies...")

# Check if we need to install anything
import subprocess
result = subprocess.run(['pip', 'show', 'xgboost'], capture_output=True, text=True)
if 'not found' in result.stderr.lower() or result.returncode != 0:
    print("   Installing additional packages...")
    !pip install -q xgboost>=2.0.3 rich>=13.7.1 python-dotenv>=1.0.0 structlog>=24.1.0
else:
    print("   ✓ Packages already installed")

# Verify key packages
import tensorflow as tf
import numpy as np
import pandas as pd

print(f"\n✅ Dependencies ready!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   GPU: {tf.config.list_physical_devices('GPU')}")

📦 Checking/Installing dependencies...
   ✓ Packages already installed

✅ Dependencies ready!
   TensorFlow: 2.19.0
   NumPy: 1.26.4
   Pandas: 2.2.0
   GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================
# 1.3 Verify TensorFlow CUDA setup & Optimize for A100
# ============================================================
import tensorflow as tf

print("=" * 70)
print("🔧 TensorFlow Configuration")
print("=" * 70)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

# Enable memory growth to prevent OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✅ Memory growth enabled for {len(gpus)} GPU(s)")
        
        # Get GPU details
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        gpu_name = gpu_details.get('device_name', 'Unknown')
        print(f"   GPU: {gpu_name}")
        
        # A100-specific optimizations
        if 'A100' in gpu_name:
            print(f"\n🚀 A100 POWERHOUSE MODE ENABLED!")
            print(f"   • 80GB VRAM → 4x larger model (d_model=128, 4 layers)")
            print(f"   • batch_size=256 (4x Mac M1)")
            print(f"   • seq_len=128 (2x temporal context)")
            print(f"   • TF32 enabled for Tensor Core acceleration")
            # Enable TF32 for A100 (faster than FP32, same accuracy)
            tf.config.experimental.enable_tensor_float_32_execution(True)
    except RuntimeError as e:
        print(f"⚠️ Could not set memory growth: {e}")

# ⚠️ MIXED PRECISION DISABLED - causes 0% accuracy bug in TF 2.19
# TF32 alone provides ~1.5x speedup without this issue
print(f"\n✅ Using float32 precision (TF32 enabled for A100)")
print("   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug")

🔧 TensorFlow Configuration
TensorFlow version: 2.19.0
CUDA available: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
⚠️ Could not set memory growth: Physical devices cannot be modified after being initialized

✅ Using float32 precision (TF32 enabled for A100)
   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug


## 2️⃣ Clone Repository & Setup

In [86]:
# ============================================================
# 2.1 Clone the ML Engine repository
# ============================================================
# ⚠️ RE-RUN THIS CELL if you see dimension mismatch errors!
# This pulls the latest code with bug fixes from GitHub.

import os

REPO_URL = "https://github.com/Raynergy-svg/ml_engine.git"
REPO_DIR = "/content/ml_engine"

# IMPORTANT: Reset to /content first (fixes "getcwd" errors after rm -rf)
os.chdir("/content")

# Remove existing directory if it exists (forces fresh clone)
if os.path.exists(REPO_DIR):
    print("🗑️ Removing existing repo to get latest fixes...")
    !rm -rf {REPO_DIR}

print(f"📥 Cloning repository from {REPO_URL}...")
!git clone {REPO_URL} {REPO_DIR}

# Change to repo directory
os.chdir(REPO_DIR)
print(f"\n📂 Working directory: {os.getcwd()}")

# Show latest commit to verify we have the fix
print("\n📋 Latest commit:")
!git log --oneline -3

print("\n📁 Repository contents:")
!ls -la

🗑️ Removing existing repo to get latest fixes...
📥 Cloning repository from https://github.com/Raynergy-svg/ml_engine.git...
Cloning into '/content/ml_engine'...
remote: Enumerating objects: 1673, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1673 (delta 79), reused 76 (delta 61), pack-reused 1570 (from 3)
Receiving objects: 100% (1673/1673), 352.35 MiB | 16.25 MiB/s, done.
Resolving deltas: 100% (843/843), done.
Updating files: 100% (504/504), done.

📂 Working directory: /content/ml_engine

📋 Latest commit:
9177110 (HEAD -> main, origin/main, origin/HEAD) feat: Update all notebook cells to match A100 Powerhouse config
7749c72 feat: A100 Powerhouse - 4x larger Transformer model
c777659 feat: A100 powerhouse config - bigger, better model

📁 Repository contents:
total 72696
drwxr-xr-x 15 root root     4096 Jan  6 17:55  .
drwxr-xr-x  1 root root     4096 Jan  6 17:55  ..
-rw-r--r--  1 root root    18969 Jan  6 17:55  a

In [87]:
# ============================================================
# 2.2 Create necessary directories & Clear stale replay buffers
# ============================================================
import os
import glob
from pathlib import Path

directories = [
    "trained_data/models",
    "trained_data/checkpoints",
    "trained_data/checkpoints/tensorflow",
    "trained_data/replay/EUR_USD",
    "trained_data/replay/USD_JPY",
    "trained_data/replay/GBP_USD",
    "trained_data/logs",
    "trained_data/scalers",
    "market_data",
]

for d in directories:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {d}")

# Clear any stale replay buffers (they may have different feature dimensions)
replay_files = glob.glob("trained_data/replay/**/*.npz", recursive=True)
if replay_files:
    print(f"\n🗑️ Clearing {len(replay_files)} stale replay buffer(s)...")
    for f in replay_files:
        os.remove(f)
        print(f"   Removed: {f}")
    print("✅ Replay buffers cleared (prevents feature dimension mismatch)")
else:
    print("\n✅ No stale replay buffers to clear")

print("\n📁 Directory structure ready!")

✅ Created: trained_data/models
✅ Created: trained_data/checkpoints
✅ Created: trained_data/checkpoints/tensorflow
✅ Created: trained_data/replay/EUR_USD
✅ Created: trained_data/replay/USD_JPY
✅ Created: trained_data/replay/GBP_USD
✅ Created: trained_data/logs
✅ Created: trained_data/scalers
✅ Created: market_data

🗑️ Clearing 2 stale replay buffer(s)...
   Removed: trained_data/replay/USD_JPY/buffer.npz
   Removed: trained_data/replay/EUR_USD/buffer.npz
✅ Replay buffers cleared (prevents feature dimension mismatch)

📁 Directory structure ready!


## 3️⃣ Environment Variables (OANDA API)

In [9]:
# ============================================================
# 3.1 Set OANDA API credentials
# ============================================================
import os
from getpass import getpass

print("=" * 70)
print("🔐 OANDA API Configuration")
print("=" * 70)
print("\nEnter your OANDA Practice account credentials.")
print("(These are stored only in this session's memory)\n")

# Interactive input with clear format hints
print("API Token format: xxxx-xxxx (long string with hyphen)")
OANDA_API_TOKEN = getpass("OANDA API Token: ")

print("\nAccount ID format: 101-001-XXXXXXXX-001")
OANDA_ACCOUNT_ID = input("OANDA Account ID: ")

# Validate inputs
if '-' in OANDA_ACCOUNT_ID and len(OANDA_ACCOUNT_ID) > 50:
    print("\n⚠️ WARNING: Account ID looks like a token - you may have swapped them!")
    print("   Swapping automatically...")
    OANDA_API_TOKEN, OANDA_ACCOUNT_ID = OANDA_ACCOUNT_ID, OANDA_API_TOKEN

# Set environment variables
os.environ["OANDA_API_TOKEN"] = OANDA_API_TOKEN
os.environ["OANDA_ACCOUNT_ID"] = OANDA_ACCOUNT_ID

# Verify (show only last 4 chars of token)
print(f"\n✅ OANDA_API_TOKEN: ...{OANDA_API_TOKEN[-4:]}")
print(f"✅ OANDA_ACCOUNT_ID: {OANDA_ACCOUNT_ID}")

🔐 OANDA API Configuration

Enter your OANDA Practice account credentials.
(These are stored only in this session's memory)

API Token format: xxxx-xxxx (long string with hyphen)

Account ID format: 101-001-XXXXXXXX-001

✅ OANDA_API_TOKEN: ...1a62
✅ OANDA_ACCOUNT_ID: 101-001-37949116-001


In [88]:
# ============================================================
# 3.2 Test OANDA connection
# ============================================================
import sys
sys.path.insert(0, '/content/ml_engine')

try:
    from oanda_practice import OandaPracticeClient
    
    client = OandaPracticeClient.from_env()
    print("✅ OANDA client initialized successfully!")
    print("\n📊 Testing candle fetch...")
    
    # Fetch a small sample to verify connection
    resp = client.get_candles(
        instrument="EUR_USD",
        granularity="H1",
        count=10
    )
    # Response is a dict with "candles" key
    candles = resp.get("candles", []) if isinstance(resp, dict) else []
    
    print(f"✅ Fetched {len(candles)} candles from OANDA")
    if candles:
        c = candles[-1]
        print(f"   Latest: {c['time']} Close: {c['mid']['c']}")
    
except Exception as e:
    print(f"❌ OANDA connection failed: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️ You can still train using local CSV files.")
    print("   Upload your market data to /content/ml_engine/market_data/")

✅ OANDA client initialized successfully!

📊 Testing candle fetch...
✅ Fetched 10 candles from OANDA
   Latest: 2026-01-06T17:00:00.000000000Z Close: 1.16998


## 4️⃣ Data Preparation

In [ ]:
# ============================================================
# 4.1 Calculate Optimal Data Size for A100 Powerhouse
# ============================================================
import numpy as np

def calculate_optimal_candles(config: dict) -> dict:
    """
    Calculate optimal candle count based on model size and A100 capacity.
    
    Rule of thumb for deep learning:
    - Minimum: 10x parameters per class for classification
    - Recommended: 50-100x for robust generalization
    - A100 80GB can handle much larger datasets in memory
    """
    # Calculate Transformer parameters
    d_model = config.get("transformer_d_model", 128)
    num_heads = config.get("transformer_num_heads", 8)
    num_layers = config.get("transformer_num_layers", 4)
    dff = config.get("transformer_dff", 512)
    seq_len = config.get("seq_len", 128)
    n_features = 80  # Approximate feature count
    
    # Transformer parameter count (approximate)
    # Embedding: features * d_model
    embed_params = n_features * d_model
    
    # Per layer: attention (4 * d_model^2) + FFN (2 * d_model * dff) + norms
    attn_params = 4 * d_model * d_model  # Q, K, V, O projections
    ffn_params = 2 * d_model * dff       # Two dense layers
    norm_params = 4 * d_model            # LayerNorm
    layer_params = attn_params + ffn_params + norm_params
    
    # Total
    total_params = embed_params + (num_layers * layer_params) + d_model  # + output
    
    # Calculate optimal samples
    min_samples = total_params * 10      # Minimum: 10x params
    recommended_samples = total_params * 50  # Recommended: 50x params
    max_efficient = total_params * 100   # Diminishing returns after this
    
    # Convert to candles (account for seq_len and train/val/test split)
    # Effective samples = (candles - seq_len) * 0.8 (train split)
    def samples_to_candles(samples):
        return int((samples / 0.8) + seq_len + 100)  # +100 buffer
    
    # A100 memory constraints (80GB)
    # Rough estimate: 4 bytes/float * features * seq_len * batch_size * 3 (activations)
    batch_size = config.get("batch_size", 256)
    bytes_per_sample = 4 * n_features * seq_len * 3  # ~150KB per sample in batch
    max_batch_memory = batch_size * bytes_per_sample  # ~38MB per batch
    # A100 can handle ~50GB for data, rest for model/gradients
    max_candles_memory = int(50e9 / (4 * n_features))  # ~150M candles (not a constraint)
    
    return {
        "model_params": total_params,
        "min_candles": samples_to_candles(min_samples),
        "recommended_candles": samples_to_candles(recommended_samples),
        "max_efficient_candles": samples_to_candles(max_efficient),
        "a100_memory_limit": min(max_candles_memory, 500000),  # Cap at 500K
        "details": {
            "d_model": d_model,
            "layers": num_layers,
            "dff": dff,
            "seq_len": seq_len,
        }
    }

# Calculate for A100 Powerhouse config
optimal = calculate_optimal_candles(TRAINING_CONFIG)

print("=" * 70)
print("📊 OPTIMAL DATA SIZE CALCULATOR (A100 Powerhouse)")
print("=" * 70)
print(f"\n🔧 Model Configuration:")
print(f"   d_model={optimal['details']['d_model']}, layers={optimal['details']['layers']}")
print(f"   dff={optimal['details']['dff']}, seq_len={optimal['details']['seq_len']}")
print(f"\n📈 Model Parameters: ~{optimal['model_params']:,}")
print(f"\n📊 Recommended Candle Counts:")
print(f"   ├─ Minimum (10x params):     {optimal['min_candles']:,} candles")
print(f"   ├─ Recommended (50x params): {optimal['recommended_candles']:,} candles  ⭐")
print(f"   ├─ Max Efficient (100x):     {optimal['max_efficient_candles']:,} candles")
print(f"   └─ A100 Memory Limit:        {optimal['a100_memory_limit']:,} candles")

# Set optimal target
OPTIMAL_CANDLES = optimal['recommended_candles']
print(f"\n✅ Using OPTIMAL_CANDLES = {OPTIMAL_CANDLES:,} (50x model params)")
print(f"   This is {OPTIMAL_CANDLES/12000:.1f}x more than the previous 12,000!")

📊 Fetching 12000 H1 candles for EUR_USD...
   This will take a few minutes for large requests...

   Fetched batch: 5000 candles | Total: 5000
   Fetched batch: 5000 candles | Total: 10000
   Fetched batch: 2000 candles | Total: 12000

✅ Saved 12000 candles to /content/ml_engine/market_data/EURUSD_H1.csv
   Date range: 2024-01-31 16:00:00+00:00 to 2026-01-06 17:00:00+00:00

📊 Data Preview:


,open,high,low,close,volume
time,,,,,
2026-01-06 13:00:00+00:00,1.17014,1.17137,1.16975,1.17110,7498
2026-01-06 14:00:00+00:00,1.17110,1.17176,1.16992,1.17092,8688
2026-01-06 15:00:00+00:00,1.17092,1.17092,1.16840,1.16953,10993
2026-01-06 16:00:00+00:00,1.16953,1.16984,1.16859,1.16881,6720
2026-01-06 17:00:00+00:00,1.16881,1.17000,1.16870,1.16999,3659


In [ ]:
# ============================================================
# 4.2 Multi-Pair Scanner & Data Fetcher
# ============================================================
# Scans top pairs and fetches optimal data for A100 training

import asyncio
from typing import List, Dict
from datetime import datetime, timedelta

# === CONFIGURATION ===
MULTI_PAIR_MODE = True  # Set to False for single pair training

# Top FX pairs by liquidity (buddy scan candidates)
SCAN_PAIRS = [
    "EUR_USD",  # Most liquid
    "USD_JPY",  # Second most liquid  
    "GBP_USD",  # Cable
    "USD_CHF",  # Swissy
    "AUD_USD",  # Aussie
    "USD_CAD",  # Loonie
    "NZD_USD",  # Kiwi
    "EUR_GBP",  # Cross
    "EUR_JPY",  # Cross
    "GBP_JPY",  # Cross (volatile)
]

GRANULARITY = "H1"  # H1 for 24-bar daily lookahead

def fetch_pair_data(client, instrument: str, target_candles: int) -> pd.DataFrame:
    """Fetch candles for a single pair with progress tracking."""
    all_candles = []
    from_time = None
    
    print(f"\n   📊 {instrument}: Fetching {target_candles:,} candles...")
    
    while len(all_candles) < target_candles:
        remaining = target_candles - len(all_candles)
        batch_size = min(5000, remaining)
        
        try:
            if from_time:
                resp = client.get_candles(
                    instrument=instrument,
                    granularity=GRANULARITY,
                    count=batch_size,
                    to_time=from_time
                )
            else:
                resp = client.get_candles(
                    instrument=instrument,
                    granularity=GRANULARITY,
                    count=batch_size
                )
            
            candles = resp.get("candles", []) if isinstance(resp, dict) else []
            
            if not candles:
                print(f"      ⚠️ No more data available ({len(all_candles):,} fetched)")
                break
                
            all_candles = candles + all_candles
            from_time = candles[0]['time']
            
            # Progress
            pct = min(100, len(all_candles) / target_candles * 100)
            print(f"      Progress: {len(all_candles):,}/{target_candles:,} ({pct:.0f}%)", end='\r')
            
        except Exception as e:
            print(f"      ❌ Error: {e}")
            break
    
    if not all_candles:
        return None
    
    # Convert to DataFrame
    df = pd.DataFrame([{
        'time': c['time'],
        'open': float(c['mid']['o']),
        'high': float(c['mid']['h']),
        'low': float(c['mid']['l']),
        'close': float(c['mid']['c']),
        'volume': int(c['volume']),
        'instrument': instrument
    } for c in all_candles])
    
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.sort_index(inplace=True)
    
    print(f"      ✅ Got {len(df):,} candles ({df.index[0].date()} to {df.index[-1].date()})")
    
    return df

def buddy_scan_pairs(pairs: List[str], client) -> List[Dict]:
    """
    Scan pairs for trading suitability (simplified buddy scan).
    Returns pairs ranked by volatility and trend strength.
    """
    print("=" * 70)
    print("🔍 BUDDY SCAN: Analyzing pairs for training suitability...")
    print("=" * 70)
    
    results = []
    
    for pair in pairs:
        try:
            # Fetch recent data for analysis
            resp = client.get_candles(
                instrument=pair,
                granularity="H1",
                count=500  # ~3 weeks for quick scan
            )
            candles = resp.get("candles", []) if isinstance(resp, dict) else []
            
            if len(candles) < 100:
                continue
            
            # Calculate metrics
            closes = np.array([float(c['mid']['c']) for c in candles])
            highs = np.array([float(c['mid']['h']) for c in candles])
            lows = np.array([float(c['mid']['l']) for c in candles])
            
            # Volatility (ATR-like)
            ranges = highs - lows
            avg_range = np.mean(ranges)
            volatility = avg_range / np.mean(closes) * 100  # As percentage
            
            # Trend strength (price change / volatility)
            price_change = abs(closes[-1] - closes[0]) / closes[0] * 100
            trend_strength = price_change / (volatility * np.sqrt(len(candles)/24))
            
            # Directional moves (% of bars with clear direction)
            returns = np.diff(closes) / closes[:-1]
            clear_moves = np.sum(np.abs(returns) > 0.0005) / len(returns) * 100
            
            # Score: balance of volatility (tradeable) and trend (predictable)
            score = volatility * 0.3 + trend_strength * 0.4 + clear_moves * 0.003
            
            results.append({
                'pair': pair,
                'volatility': volatility,
                'trend_strength': trend_strength,
                'clear_moves_pct': clear_moves,
                'score': score,
                'current_price': closes[-1]
            })
            
            print(f"   {pair}: vol={volatility:.3f}%, trend={trend_strength:.2f}, score={score:.3f}")
            
        except Exception as e:
            print(f"   {pair}: ❌ Error - {e}")
    
    # Sort by score
    results.sort(key=lambda x: x['score'], reverse=True)
    
    print(f"\n🏆 Top pairs for training:")
    for i, r in enumerate(results[:5], 1):
        print(f"   {i}. {r['pair']} (score: {r['score']:.3f})")
    
    return results

# === RUN BUDDY SCAN ===
if MULTI_PAIR_MODE:
    scan_results = buddy_scan_pairs(SCAN_PAIRS, client)
    
    # Select top N pairs for training
    TOP_N_PAIRS = 5  # Train on top 5 pairs
    SELECTED_PAIRS = [r['pair'] for r in scan_results[:TOP_N_PAIRS]]
    
    print(f"\n✅ Selected {len(SELECTED_PAIRS)} pairs for multi-pair training:")
    print(f"   {', '.join(SELECTED_PAIRS)}")
else:
    SELECTED_PAIRS = ["EUR_USD"]
    print(f"ℹ️ Single-pair mode: {SELECTED_PAIRS[0]}")

In [ ]:
# ============================================================
# 4.3 Fetch Optimal Data for All Selected Pairs
# ============================================================
import os

# Calculate per-pair candles (split total across pairs)
if MULTI_PAIR_MODE:
    # For multi-pair: distribute optimal candles across pairs
    # More data per pair = better, but also train on diversity
    CANDLES_PER_PAIR = max(OPTIMAL_CANDLES // len(SELECTED_PAIRS), 15000)
    print(f"\n📊 Multi-pair fetching: {CANDLES_PER_PAIR:,} candles per pair")
    print(f"   Total data: ~{CANDLES_PER_PAIR * len(SELECTED_PAIRS):,} candles across {len(SELECTED_PAIRS)} pairs")
else:
    CANDLES_PER_PAIR = OPTIMAL_CANDLES

# Fetch data for each pair
pair_dataframes = {}
DATA_PATHS = {}

print("\n" + "=" * 70)
print("📥 FETCHING DATA FOR A100 TRAINING")
print("=" * 70)

for pair in SELECTED_PAIRS:
    df = fetch_pair_data(client, pair, CANDLES_PER_PAIR)
    
    if df is not None and len(df) > 1000:  # Minimum viable data
        pair_dataframes[pair] = df
        
        # Save to disk
        safe_name = pair.replace('_', '')
        data_path = f"/content/ml_engine/market_data/{safe_name}_{GRANULARITY}.csv"
        df.to_csv(data_path)
        DATA_PATHS[pair] = data_path
        
        print(f"   💾 Saved: {data_path}")
    else:
        print(f"   ⚠️ Skipping {pair} - insufficient data")

print(f"\n✅ Fetched data for {len(pair_dataframes)} pairs:")
for pair, df in pair_dataframes.items():
    print(f"   • {pair}: {len(df):,} candles")

# Total candles
TOTAL_CANDLES = sum(len(df) for df in pair_dataframes.values())
print(f"\n📊 Total training data: {TOTAL_CANDLES:,} candles")
print(f"   This is {TOTAL_CANDLES/optimal['recommended_candles']*100:.0f}% of optimal for A100 powerhouse")

In [ ]:
# ============================================================
# 4.4 Combine Multi-Pair Data for Training
# ============================================================
# Combines all pairs into unified training dataset with pair embeddings

def prepare_multi_pair_data(pair_dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Combine multiple pair DataFrames into unified training dataset.
    Adds pair identifier for potential pair-specific learning.
    """
    combined_dfs = []
    
    for pair, df in pair_dfs.items():
        df = df.copy()
        
        # Add pair identifier (for potential embedding)
        df['pair'] = pair
        df['pair_id'] = list(pair_dfs.keys()).index(pair)
        
        # Normalize OHLCV relative to close (makes pairs comparable)
        df['norm_open'] = df['open'] / df['close']
        df['norm_high'] = df['high'] / df['close']
        df['norm_low'] = df['low'] / df['close']
        df['norm_close'] = 1.0  # Always 1 after normalization
        
        # Returns (universal across pairs)
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
        
        combined_dfs.append(df)
    
    # Concatenate all pairs
    combined = pd.concat(combined_dfs, axis=0)
    
    # Sort by time (interleaves pairs)
    combined = combined.sort_index()
    
    # Remove NaN from returns calculation
    combined = combined.dropna()
    
    return combined

if MULTI_PAIR_MODE and len(pair_dataframes) > 1:
    print("=" * 70)
    print("🔀 COMBINING MULTI-PAIR DATA")
    print("=" * 70)
    
    combined_df = prepare_multi_pair_data(pair_dataframes)
    
    print(f"\n📊 Combined dataset statistics:")
    print(f"   Total samples: {len(combined_df):,}")
    print(f"   Date range: {combined_df.index.min()} to {combined_df.index.max()}")
    print(f"\n   Samples per pair:")
    for pair in SELECTED_PAIRS:
        pair_count = len(combined_df[combined_df['pair'] == pair])
        print(f"      {pair}: {pair_count:,}")
    
    # Save combined data
    COMBINED_DATA_PATH = "/content/ml_engine/market_data/MULTI_PAIR_H1.csv"
    combined_df.to_csv(COMBINED_DATA_PATH)
    print(f"\n   💾 Saved: {COMBINED_DATA_PATH}")
    
    # Use combined for training
    DATA_PATH = COMBINED_DATA_PATH
    CANDLES = len(combined_df)
    INSTRUMENT = "MULTI_PAIR"
    
    print(f"\n✅ Multi-pair data ready: {CANDLES:,} samples")
else:
    # Single pair mode - use first pair
    INSTRUMENT = SELECTED_PAIRS[0]
    DATA_PATH = DATA_PATHS.get(INSTRUMENT)
    CANDLES = len(pair_dataframes.get(INSTRUMENT, pd.DataFrame()))
    print(f"ℹ️ Single-pair mode: {INSTRUMENT} ({CANDLES:,} candles)")

In [ ]:
# ============================================================
# 4.5 Data Preview and Validation (Multi-Pair Aware)
# ============================================================
import pandas as pd

# Check if DATA_PATH was set by previous cells
if 'DATA_PATH' not in dir() or DATA_PATH is None:
    # Try to find existing data file
    import glob
    csv_files = glob.glob("/content/ml_engine/market_data/*.csv")
    if csv_files:
        # Prefer multi-pair if available
        multi_path = "/content/ml_engine/market_data/MULTI_PAIR_H1.csv"
        if multi_path in csv_files:
            DATA_PATH = multi_path
        else:
            DATA_PATH = csv_files[0]
        print(f"📁 Using existing data: {DATA_PATH}")
    else:
        print("❌ No data file found. Run cells 4.1-4.4 first to fetch data,")
        print("   or upload CSV files to /content/ml_engine/market_data/")
        DATA_PATH = None

if DATA_PATH:
    df = pd.read_csv(DATA_PATH)
    CANDLES = len(df)
    
    print("=" * 70)
    print("📊 DATA PREVIEW")
    print("=" * 70)
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    
    # Check if multi-pair data
    if 'pair' in df.columns:
        print("\n🔀 MULTI-PAIR DATASET DETECTED!")
        print(f"\n   Pairs included:")
        for pair in df['pair'].unique():
            pair_count = len(df[df['pair'] == pair])
            print(f"      • {pair}: {pair_count:,} candles")
    
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nLast 5 rows:")
    display(df.tail())
    print(f"\nStatistics:")
    display(df.describe())
    
    # Check for NaN values
    nan_counts = df.isna().sum()
    if nan_counts.any():
        print(f"\n⚠️ NaN values detected:")
        print(nan_counts[nan_counts > 0])
    else:
        print(f"\n✅ No NaN values in data")
    
    # A100 data sufficiency check
    if 'optimal' in dir():
        sufficiency = CANDLES / optimal['recommended_candles'] * 100
        print(f"\n📊 A100 Data Sufficiency: {sufficiency:.0f}%")
        if sufficiency < 50:
            print(f"   ⚠️ Consider fetching more data for optimal A100 training")
        elif sufficiency >= 100:
            print(f"   ✅ Sufficient data for A100 powerhouse training!")

📊 Data Preview
Shape: (12000, 6)

Columns: ['time', 'open', 'high', 'low', 'close', 'volume']

First 5 rows:


,time,open,high,low,close,volume
0,2024-01-31 16:00:00+00:00,1.08614,1.08680,1.08448,1.08545,7785
1,2024-01-31 17:00:00+00:00,1.08546,1.08546,1.08420,1.08498,4161
2,2024-01-31 18:00:00+00:00,1.08497,1.08526,1.08438,1.08466,4255
3,2024-01-31 19:00:00+00:00,1.08464,1.08644,1.08159,1.08405,21419
4,2024-01-31 20:00:00+00:00,1.08404,1.08419,1.07950,1.08076,16350



Last 5 rows:


,time,open,high,low,close,volume
11995,2026-01-06 13:00:00+00:00,1.17014,1.17137,1.16975,1.17110,7498
11996,2026-01-06 14:00:00+00:00,1.17110,1.17176,1.16992,1.17092,8688
11997,2026-01-06 15:00:00+00:00,1.17092,1.17092,1.16840,1.16953,10993
11998,2026-01-06 16:00:00+00:00,1.16953,1.16984,1.16859,1.16881,6720
11999,2026-01-06 17:00:00+00:00,1.16881,1.16962,1.16870,1.16938,2746



Statistics:


,open,high,low,close,volume
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.00000
mean,1.107273,1.107932,1.106629,1.107283,5454.08900
std,0.044835,0.044854,0.044822,0.044837,4770.77166
min,1.019340,1.020410,1.017790,1.019340,95.00000
25%,1.076680,1.077150,1.076130,1.076680,2386.50000
50%,1.092465,1.092990,1.091930,1.092495,4215.00000
75%,1.156400,1.157163,1.155590,1.156405,7016.75000
max,1.187110,1.191880,1.186600,1.187100,59068.00000



✅ No NaN values in data


## 5️⃣ Training Configuration (CUDA-Optimized)

In [ ]:
# ============================================================
# 5.1 Training hyperparameters - A100 POWERHOUSE CONFIG
# ============================================================
# Based on NVIDIA A100 Tensor Core optimization research:
# - A100 uses 64-element alignment (not 8!) for maximum efficiency
# - Larger models train efficiently on 80GB VRAM
# - Bigger batch sizes with gradient accumulation for stability

TRAINING_CONFIG = {
    # === MODEL ARCHITECTURE - SCALED FOR A100 ===
    # All dimensions are multiples of 64 for A100 Tensor Core optimization
    "model_type": "ensemble",
    
    # Transformer - 4x LARGER than Mac M1
    "transformer_d_model": 128,      # 32→128 (4x) - more representational capacity
    "transformer_num_heads": 8,      # 4→8 (2x) - more attention patterns
    "transformer_num_layers": 4,     # 2→4 (2x) - deeper network
    "transformer_dff": 512,          # 64→512 (8x) - wider feedforward
    "transformer_dropout": 0.15,     # Slightly less dropout for larger model
    
    # === A100-OPTIMIZED TRAINING ===
    "epochs": 200,
    "batch_size": 256,               # 64→256 (4x) - A100 handles this easily
    "learning_rate": 0.0003,         # Keep proven LR (sqrt scaling not needed with warmup)
    "patience": 25,                  # More patience for larger model
    "seq_len": 128,                  # 64→128 (2x) - more temporal context
    
    # === DIRECTION PREDICTION - PROVEN SETTINGS ===
    "direction_threshold": 0.0015,   # 0.15% - filters noise (KEEP FROM MAC)
    "direction_lookahead": 24,       # 24 bars = 1 day (KEEP FROM MAC)
    
    # === A100 TENSOR CORE ACCELERATION ===
    "use_tf32": True,                # TF32 for float32 ops (1.5x speedup)
    "jit_compile": True,             # XLA compilation (graph optimization)
    "steps_per_execution": 32,       # Reduce Python overhead
    
    # === LEARNING RATE SCHEDULE ===
    "warmup_epochs": 5,              # Warm up LR for stability with large batch
    "use_cosine_decay": True,        # Cosine annealing
    
    # === CONTINUAL LEARNING ===
    "use_ema": True,
    "ema_decay": 0.999,
    "use_ewc": True,
    "ewc_lambda": 1000.0,
    "use_replay_buffer": True,
    "replay_buffer_ratio": 0.10,
    
    # === WALK-FORWARD VALIDATION ===
    "cv_folds": 3,
    "min_train_samples": 4000,
    "test_period": 1000,
    "gap": 24,
    
    # === OVERFITTING PREVENTION ===
    "enable_swa": True,
    "enable_cosine_restarts": True,
    "overfit_threshold": 0.08,
    "critical_threshold": 0.15,
    "max_acceptable_gap": 0.12,
}

print("=" * 70)
print("🚀 A100 POWERHOUSE CONFIG")
print("=" * 70)
print("\n📊 Model Scaling (vs Mac M1):")
print("   ┌──────────────────┬─────────┬─────────┬────────┐")
print("   │ Parameter        │ Mac M1  │ A100    │ Scale  │")
print("   ├──────────────────┼─────────┼─────────┼────────┤")
print("   │ d_model          │ 32      │ 128     │ 4x     │")
print("   │ num_heads        │ 4       │ 8       │ 2x     │")
print("   │ num_layers       │ 2       │ 4       │ 2x     │")
print("   │ dff              │ 64      │ 512     │ 8x     │")
print("   │ seq_len          │ 64      │ 128     │ 2x     │")
print("   │ batch_size       │ 64      │ 256     │ 4x     │")
print("   └──────────────────┴─────────┴─────────┴────────┘")
print(f"\n🎯 Proven Settings (kept from Mac):")
print(f"   • lookahead:  {TRAINING_CONFIG['direction_lookahead']} bars (1-day prediction)")
print(f"   • threshold:  {TRAINING_CONFIG['direction_threshold']*100:.2f}% (noise filter)")
print(f"   • LR:         {TRAINING_CONFIG['learning_rate']} (with warmup)")
print("\n⚡ A100 Optimizations:")
print("   • All dims multiples of 64 (Tensor Core aligned)")
print("   • TF32 enabled for matrix ops")
print("   • XLA compilation for fused kernels")
print("\n✅ Expected: ~60-65% balanced accuracy (larger model = better patterns)")

⚙️ Training Configuration (A100 80GB FULLY OPTIMIZED)

🚀 A100 Tensor Core Optimizations Applied:
   ┌─────────────────────────────────────────────────────────────┐
   │ Parameter          │ Before    │ After     │ Reason        │
   ├─────────────────────────────────────────────────────────────┤
   │ batch_size         │ 256       │ 512       │ 80GB VRAM     │
   │ d_model            │ 32        │ 64        │ Multiple of 8 │
   │ num_heads          │ 4         │ 8         │ Multiple of 8 │
   │ dff                │ 64        │ 128       │ Multiple of 8 │
   │ seq_len            │ 60        │ 64        │ Multiple of 8 │
   │ steps_per_execution│ 20        │ 32        │ Less overhead │
   │ learning_rate      │ 0.0003    │ 0.0005    │ sqrt(batch)   │
   └─────────────────────────────────────────────────────────────┘

📊 Expected Performance Gains:
   • Mixed Precision (FP16): ~2x speedup on Tensor Cores
   • TF32 for FP32 ops: ~1.5x speedup (A100 exclusive)
   • XLA Compilation: ~1.2-1.5x

## 6️⃣ Run Training

In [73]:
# ============================================================
# 6.1 Import training modules
# ============================================================
import os
import sys
import logging

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Add repo to path
sys.path.insert(0, '/content/ml_engine')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("📦 Importing training modules...")

import numpy as np
import pandas as pd
import tensorflow as tf

# Import trainers (correct class names!)
from modular_trainers import (
    TrainerConfig,
    TransformerDirectionTrainer,
    XGBoostTrainer,          # NOT XGBoostMomentumTrainer
    RandomForestTrainer,     # NOT RandomForestRiskTrainer
    RidgeTrainer,            # NOT RidgeConfidenceTrainer
    OverfitPreventionCallback,
)

# Import data loaders (correct function names!)
from modular_data_loaders import (
    compute_normalized_features,
    load_direction_data,      # NOT prepare_direction_data
    load_xgboost_data,        # NOT prepare_momentum_data
    load_rf_data,             # NOT prepare_risk_data
    load_ridge_data,          # NOT prepare_confidence_data
)

# Import feature engineering
from feature_engineering import FeatureEngineering

print("✅ All modules imported successfully!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   GPU devices: {tf.config.list_physical_devices('GPU')}")

📦 Importing training modules...
✅ All modules imported successfully!
   TensorFlow: 2.19.0
   GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [74]:
# ============================================================
# 6.2 Prepare training data
# ============================================================
from rich.console import Console
from rich.panel import Panel

console = Console()

console.print(Panel("[bold blue]Step 1: Data Preparation[/bold blue]"))

# Load data
df = pd.read_csv(DATA_PATH)
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

# Rename columns to lowercase
df.columns = [c.lower() for c in df.columns]

console.print(f"  📊 Loaded {len(df)} candles")
console.print(f"  📅 Date range: {df.index.min()} to {df.index.max()}")

# Compute normalized features
console.print("  🔧 Computing normalized features...")
df = compute_normalized_features(df)

# Add technical indicators
fe = FeatureEngineering()
df = fe.add_technical_indicators(df)

# Fill NaN values
df = df.ffill().bfill()

# Drop remaining NaN rows
df = df.dropna()

console.print(f"  ✅ Features computed: {len(df.columns)} columns")
console.print(f"  ✅ Clean rows: {len(df)}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 1: Data Preparation                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Loaded 12000 candles

📅 Date range: 2024-01-31 16:00:00+00:00 to 2026-01-06 17:00:00+00:00

🔧 Computing normalized features...

✅ Features computed: 117 columns

✅ Clean rows: 12000

In [ ]:
# ============================================================
# 6.3 Prepare model-specific datasets
# ============================================================
console.print(Panel("[bold blue]Step 2: Prepare Model-Specific Data[/bold blue]"))

SEQ_LEN = TRAINING_CONFIG["seq_len"]           # 128 (2x Mac for more context)
LOOKAHEAD = TRAINING_CONFIG["direction_lookahead"]  # 24 bars (1 day) - proven setting
THRESHOLD = TRAINING_CONFIG["direction_threshold"]   # 0.15% - filters noise

# Direction data (Transformer) - uses load_direction_data
console.print("  📊 Preparing Direction data (Transformer)...")
console.print(f"     seq_len={SEQ_LEN} (A100: 2x temporal context)")
console.print(f"     lookahead={LOOKAHEAD} bars, threshold={THRESHOLD*100:.2f}%")

direction_data = load_direction_data(
    df, 
    split=(0.8, 0.1, 0.1), 
    lookahead=LOOKAHEAD,     # 24 bars = 1 day prediction
    threshold=THRESHOLD       # 0.15% filters noise
)
console.print(f"     X_train: {direction_data['X_train'].shape}")
console.print(f"     y_train: {direction_data['y_train'].shape}")

# Check class distribution
y_train_direction = direction_data['y_train']
n_up = (y_train_direction == 1).sum()
n_down = (y_train_direction == 0).sum()
n_unclear = ((y_train_direction != 0) & (y_train_direction != 1)).sum()
total = len(y_train_direction)

console.print(f"     📊 Class distribution: UP={n_up} ({100*n_up/total:.1f}%), DOWN={n_down} ({100*n_down/total:.1f}%)")
if n_unclear > 0:
    console.print(f"     ⚠️ Unclear samples filtered: {n_unclear} ({100*n_unclear/total:.1f}%)")

# Check imbalance ratio
if n_up > 0 and n_down > 0:
    imbalance = max(n_up, n_down) / min(n_up, n_down)
    if imbalance > 2.0:
        console.print(f"     ⚠️ High imbalance: {imbalance:.2f}x - class weights will be applied")
    else:
        console.print(f"     ✅ Balanced classes (imbalance: {imbalance:.2f}x)")

# Momentum data (XGBoost) - uses load_xgboost_data
console.print("  📊 Preparing Momentum data (XGBoost)...")
momentum_data = load_xgboost_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {momentum_data['X_train'].shape}")

# Risk data (Random Forest) - uses load_rf_data
console.print("  📊 Preparing Risk data (Random Forest)...")
risk_data = load_rf_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {risk_data['X_train'].shape}")

# Confidence data (Ridge) - uses load_ridge_data
console.print("  📊 Preparing Confidence data (Ridge)...")
confidence_data = load_ridge_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {confidence_data['X_train'].shape}")

console.print("\n✅ All datasets prepared (A100 Powerhouse config)!")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 2: Prepare Model-Specific Data                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Preparing Direction data (Transformer)...

X_train: (9595, 50)

y_train: (9595,)

📊 Class distribution: UP=4984 (51.9%), DOWN=4611 (48.1%)

...

📊 Preparing Risk data (Random Forest)...

X_train: (9592, 18)

📊 Preparing Confidence data (Ridge)...

X_train: (9592, 14)

✅ All datasets prepared!

In [76]:
# ============================================================
# 6.3.1 DIAGNOSTIC: Check data quality for each model
# ============================================================
console.print(Panel("[bold yellow]🔍 DATA QUALITY DIAGNOSTICS[/bold yellow]"))

def check_array_quality(name, X, y=None):
    """Check for NaN, Inf, constant values, scale issues."""
    issues = []
    
    # Check X
    if np.isnan(X).any():
        nan_pct = 100 * np.isnan(X).sum() / X.size
        issues.append(f"X has {nan_pct:.1f}% NaN")
    if np.isinf(X).any():
        inf_pct = 100 * np.isinf(X).sum() / X.size
        issues.append(f"X has {inf_pct:.1f}% Inf")
    
    # Check for constant columns
    if X.ndim == 2:
        const_cols = np.sum(np.std(X, axis=0) < 1e-10)
        if const_cols > 0:
            issues.append(f"{const_cols}/{X.shape[1]} constant features")
    
    # Check scale
    x_mean = np.nanmean(np.abs(X))
    x_max = np.nanmax(np.abs(X))
    if x_max > 1e6:
        issues.append(f"X scale too large: max={x_max:.2e}")
    
    # Check y
    if y is not None:
        if np.isnan(y).any():
            nan_pct = 100 * np.isnan(y).sum() / y.size
            issues.append(f"y has {nan_pct:.1f}% NaN")
        if np.isinf(y).any():
            issues.append(f"y has Inf values")
        
        y_std = np.nanstd(y.flatten() if y.ndim > 1 else y)
        if y_std < 1e-10:
            issues.append(f"y is CONSTANT (std={y_std:.2e})")
        
        y_min, y_max = np.nanmin(y), np.nanmax(y)
        console.print(f"  y range: [{y_min:.4f}, {y_max:.4f}], std={y_std:.4f}")
    
    if issues:
        console.print(f"[red]  ❌ {name}: {', '.join(issues)}[/red]")
    else:
        console.print(f"[green]  ✅ {name}: OK (X shape={X.shape}, mean_abs={x_mean:.4f})[/green]")
    
    return len(issues) == 0

# Check each dataset
console.print("\n📊 Direction Data (Transformer):")
check_array_quality("Direction", direction_data['X_train'], direction_data['y_train'])

console.print("\n📊 Momentum Data (XGBoost):")
check_array_quality("Momentum", momentum_data['X_train'], momentum_data['y_train'])
# XGBoost y has 2 columns: [momentum_score, acceleration]
console.print(f"  y[:, 0] (momentum): range=[{momentum_data['y_train'][:, 0].min():.4f}, {momentum_data['y_train'][:, 0].max():.4f}]")
console.print(f"  y[:, 1] (accel): unique={np.unique(momentum_data['y_train'][:, 1])}")

console.print("\n📊 Risk Data (Random Forest):")
check_array_quality("Risk", risk_data['X_train'], risk_data['y_train'])
# RF y has 2 columns: [drawdown_pct, streak_prob]
console.print(f"  y[:, 0] (drawdown): range=[{risk_data['y_train'][:, 0].min():.6f}, {risk_data['y_train'][:, 0].max():.6f}]")
console.print(f"  y[:, 1] (streak): range=[{risk_data['y_train'][:, 1].min():.4f}, {risk_data['y_train'][:, 1].max():.4f}]")

console.print("\n📊 Confidence Data (Ridge):")
check_array_quality("Confidence", confidence_data['X_train'], confidence_data['y_train'])

# Summary
console.print("\n" + "="*70)
console.print("[bold]If any dataset shows issues above, that explains the bad results![/bold]")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔍 DATA QUALITY DIAGNOSTICS                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Direction Data (Transformer):

y range: [0.0000, 1.0000], std=0.4996

  ✅ Direction: OK (X shape=(9595, 50), mean_abs=4964.7549)

📊 Momentum Data (XGBoost):

...

📊 Confidence Data (Ridge):

y range: [29.1201, 100.0000], std=8.8334

  ✅ Confidence: OK (X shape=(9592, 14), mean_abs=0.2838)

======================================================================

If any dataset shows issues above, that explains the bad results!

In [ ]:
# ============================================================
# 6.4 Train Transformer (Direction Predictor) - A100 POWERHOUSE
# ============================================================
console.print(Panel("[bold green]Step 3/6: Training Transformer (Direction)[/bold green]"))

# Create trainer config with A100 POWERHOUSE settings
config = TrainerConfig(
    epochs=TRAINING_CONFIG["epochs"],
    batch_size=TRAINING_CONFIG["batch_size"],           # 256 (4x Mac)
    learning_rate=TRAINING_CONFIG["learning_rate"],
    patience=TRAINING_CONFIG["patience"],               # 25 (more patience for larger model)
    transformer_d_model=TRAINING_CONFIG["transformer_d_model"],     # 128 (4x Mac)
    transformer_num_heads=TRAINING_CONFIG["transformer_num_heads"], # 8 (2x Mac)
    transformer_num_layers=TRAINING_CONFIG["transformer_num_layers"], # 4 (2x Mac)
    transformer_dff=TRAINING_CONFIG["transformer_dff"],             # 512 (8x Mac)
    transformer_dropout=TRAINING_CONFIG["transformer_dropout"],     # 0.15
    use_ema=TRAINING_CONFIG["use_ema"],
    ema_decay=TRAINING_CONFIG["ema_decay"],
    use_ewc=TRAINING_CONFIG["use_ewc"],
    ewc_lambda=TRAINING_CONFIG["ewc_lambda"],
)

# Create and train Transformer
transformer_trainer = TransformerDirectionTrainer(config)

console.print(f"  🚀 A100 POWERHOUSE Transformer:")
console.print(f"     d_model={config.transformer_d_model} (4x Mac), heads={config.transformer_num_heads} (2x Mac)")
console.print(f"     layers={config.transformer_num_layers} (2x Mac), dff={config.transformer_dff} (8x Mac)")
console.print(f"     batch_size={config.batch_size}, seq_len={TRAINING_CONFIG['seq_len']}")

# Train
transformer_result = transformer_trainer.train(
    X_train=direction_data['X_train'],
    y_train=direction_data['y_train'],
    X_val=direction_data['X_val'],
    y_val=direction_data['y_val'],
    feature_names=direction_data.get('feature_names'),
    instrument=INSTRUMENT,
    data_range=f"{TARGET_CANDLES} candles",
)

console.print(f"\n✅ Transformer trained!")
console.print(f"   Val Accuracy: {transformer_result.get('val_accuracy', 0):.1%}")
console.print(f"   Balanced Acc: {transformer_result.get('val_balanced_accuracy', 0):.1%}")
console.print(f"   ↳ UP accuracy:   {transformer_result.get('val_up_accuracy', 0):.1%}")
console.print(f"   ↳ DOWN accuracy: {transformer_result.get('val_down_accuracy', 0):.1%}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 3/6: Training Transformer (Direction)                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🏗️ Building Transformer model...

d_model=64, heads=8

layers=2, dff=128

Training Transformer Direction for up to 200 epochs...

...

✅ Transformer trained!

Val Accuracy: 56.5%

Balanced Acc: 56.5%

↳ UP accuracy:   45.2%

↳ DOWN accuracy: 67.7%

In [78]:
# ============================================================
# 6.5 Train XGBoost (Momentum Analyzer)
# ============================================================
console.print(Panel("[bold green]Step 4/6: Training XGBoost (Momentum)[/bold green]"))

xgb_trainer = XGBoostTrainer(config)

xgb_result = xgb_trainer.train(
    X_train=momentum_data['X_train'],
    y_train=momentum_data['y_train'],
    X_val=momentum_data['X_val'],
    y_val=momentum_data['y_val'],
    feature_names=momentum_data.get('feature_names'),
)

console.print(f"\n✅ XGBoost trained!")
console.print(f"   Momentum MAE: {xgb_result.get('momentum_mae', 0):.4f}")
console.print(f"   Accel Accuracy: {xgb_result.get('acceleration_accuracy', 0):.1%}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 4/6: Training XGBoost (Momentum)                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ XGBoost trained!

Momentum MAE: 0.0330

Accel Accuracy: 86.1%

In [79]:
# ============================================================
# 6.6 Train Random Forest (Risk Assessor)
# ============================================================
console.print(Panel("[bold green]Step 5/6: Training Random Forest (Risk)[/bold green]"))

rf_trainer = RandomForestTrainer(config)

rf_result = rf_trainer.train(
    X_train=risk_data['X_train'],
    y_train=risk_data['y_train'],
    X_val=risk_data['X_val'],
    y_val=risk_data['y_val'],
    feature_names=risk_data.get('feature_names'),
)

console.print(f"\n✅ Random Forest trained!")
console.print(f"   Drawdown MAE: {rf_result.get('drawdown_mae', 0):.4f}")
console.print(f"   Streak MAE: {rf_result.get('streak_mae', 0):.4f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 5/6: Training Random Forest (Risk)                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Random Forest trained!

Drawdown MAE: 0.0000

Streak MAE: 0.0000

In [80]:
# ============================================================
# 6.7 Train Ridge (Confidence Scorer)
# ============================================================
console.print(Panel("[bold green]Step 6/6: Training Ridge (Confidence)[/bold green]"))

ridge_trainer = RidgeTrainer(config)

ridge_result = ridge_trainer.train(
    X_train=confidence_data['X_train'],
    y_train=confidence_data['y_train'],
    X_val=confidence_data['X_val'],
    y_val=confidence_data['y_val'],
    feature_names=confidence_data.get('feature_names'),
)

console.print(f"\n✅ Ridge trained!")
console.print(f"   Confidence MAE: {ridge_result.get('confidence_mae', 0):.2f}")
console.print(f"   R² Score: {ridge_result.get('r2_score', 0):.3f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 6/6: Training Ridge (Confidence)                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Ridge trained!

Confidence MAE: 2.95

R² Score: 0.485

In [81]:
# ============================================================
# 6.8 Save all models
# ============================================================
console.print(Panel("[bold blue]Saving Models[/bold blue]"))

import json
from datetime import datetime
import shutil

MODEL_DIR = "trained_data/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save Transformer (with instrument for replay buffer)
transformer_trainer.save(f"{MODEL_DIR}/transformer_direction.keras", instrument=INSTRUMENT)
console.print(f"  💾 Saved: {MODEL_DIR}/transformer_direction.keras")

# Save XGBoost
xgb_trainer.save(f"{MODEL_DIR}/xgb_momentum.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/xgb_momentum.pkl")

# Save Random Forest
rf_trainer.save(f"{MODEL_DIR}/rf_risk.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/rf_risk.pkl")

# Save Ridge
ridge_trainer.save(f"{MODEL_DIR}/ridge_confidence.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/ridge_confidence.pkl")

# Save ensemble metadata
metadata = {
    "trained_at": datetime.now().isoformat(),
    "trained_on": "colab_cuda_a100",
    "instrument": INSTRUMENT,
    "granularity": GRANULARITY,
    "candles": CANDLES,
    "config": TRAINING_CONFIG,
    "results": {
        "transformer": transformer_result,
        "xgboost": xgb_result,
        "random_forest": rf_result,
        "ridge": ridge_result,
    }
}

with open(f"{MODEL_DIR}/modular_ensemble.meta.json", 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
console.print(f"  💾 Saved: {MODEL_DIR}/modular_ensemble.meta.json")

console.print("\n✅ All models saved!")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Saving Models                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

💾 Saved: trained_data/models/transformer_direction.keras

💾 Saved: trained_data/models/xgb_momentum.pkl

💾 Saved: trained_data/models/rf_risk.pkl

💾 Saved: trained_data/models/ridge_confidence.pkl

💾 Saved: trained_data/models/modular_ensemble.meta.json

✅ All models saved!

## 7️⃣ Training Summary & Visualization

In [82]:
# ============================================================
# 7.1 Training summary
# ============================================================
from rich.table import Table

console.print(Panel("[bold green]🎉 Training Complete![/bold green]"))

summary_table = Table(title="Model Performance Summary")
summary_table.add_column("Model", style="cyan")
summary_table.add_column("Metric", style="magenta")
summary_table.add_column("Value", style="green")

# Transformer - use correct keys (val_balanced_accuracy, not balanced_accuracy)
summary_table.add_row("Transformer", "Val Accuracy", f"{transformer_result.get('val_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "Balanced Acc", f"{transformer_result.get('val_balanced_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "↳ UP Accuracy", f"{transformer_result.get('val_up_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "↳ DOWN Accuracy", f"{transformer_result.get('val_down_accuracy', 0):.1%}")

# XGBoost - correct key is 'acceleration_accuracy' not 'accel_accuracy'
summary_table.add_row("XGBoost", "Accel Accuracy", f"{xgb_result.get('acceleration_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Momentum MAE", f"{xgb_result.get('momentum_mae', 0):.4f}")

# Random Forest - correct keys: 'drawdown_mae_pct' and 'streak_prob_mae'
dd_mae = rf_result.get('drawdown_mae_pct', rf_result.get('drawdown_mae', 0))
streak_mae = rf_result.get('streak_prob_mae', rf_result.get('streak_mae', 0))
summary_table.add_row("Random Forest", "Drawdown MAE", f"{dd_mae:.6f}" if dd_mae < 0.01 else f"{dd_mae:.4f}")
summary_table.add_row("Random Forest", "Streak MAE", f"{streak_mae:.4f}")

# Ridge
summary_table.add_row("Ridge", "R² Score", f"{ridge_result.get('r2_score', 0):.3f}")
summary_table.add_row("Ridge", "Confidence MAE", f"{ridge_result.get('confidence_mae', 0):.2f}")

console.print(summary_table)

# === DIAGNOSTIC: Show actual result dicts ===
console.print("\n[yellow]🔍 Debug: Raw result dictionaries[/yellow]")
console.print(f"   transformer_result: {transformer_result}")
console.print(f"   xgb_result: {xgb_result}")
console.print(f"   rf_result: {rf_result}")  
console.print(f"   ridge_result: {ridge_result}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🎉 Training Complete!                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

          Model Performance Summary           
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Model         ┃ Metric          ┃ Value    ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Transformer   │ Val Accuracy    │ 56.5%    │
│ Transformer   │ Balanced Acc    │ 56.5%    │
│ Transformer   │ ↳ UP Accuracy   │ 45.2%    │
│ Transformer   │ ↳ DOWN Accuracy │ 67.7%    │
│ XGBoost       │ Accel Accuracy  │ 86.1%    │
│ XGBoost       │ Momentum MAE    │ 0.0330   │
│ Random Forest │ Drawdown MAE    │ 0.000662 │
│ Random Forest │ Streak MAE      │ 0.1344   │
│ Ridge         │ R² Score        │ 0.485    │
│ Ridge         │ Confidence MAE  │ 2.95     │
└───────────────┴─────────────────┴──────────┘

🔍 Debug: Raw result dictionaries

transformer_result: {'train_accuracy': 0.5374934673309326, 'val_accuracy': 0.5654082528533801, 
'val_balanced_accuracy': 0.564717345321567, 'val_up_accuracy': 0.45229681978798586, 'val_down_accuracy': 
0.6771378708551483, 'epochs_trained': 28, 'n_train_samples': 9535, 'n_val_samples': 1139, 'total_weight_norm': 
84.0878656417899, 'avg_weight_norm': 2.212838569520787, 'drift_detected': False}

xgb_result: {'momentum_mae': 0.033021699637174606, 'acceleration_accuracy': 0.8614357262103506}

rf_result: {'drawdown_mae_pct': 0.000662256436177492, 'drawdown_mae_bps': 6.62256436177492, 'streak_prob_mae': 
0.13437153927703208}

ridge_result: {'confidence_mae': 2.9538168907165527, 'r2_score': 0.4852351917048725}

## 8️⃣ Download Models

In [83]:
# ============================================================
# 8.1 Package models for download
# ============================================================
import shutil
from datetime import datetime

# Create zip file with all models
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f"ml_engine_models_{INSTRUMENT.replace('/', '_')}_{timestamp}"

# Create archive
shutil.make_archive(
    f"/content/{zip_name}",
    'zip',
    root_dir='/content/ml_engine',
    base_dir='trained_data/models'
)

print(f"✅ Models packaged: /content/{zip_name}.zip")
print(f"\n📦 Contents:")
!unzip -l /content/{zip_name}.zip | head -20

✅ Models packaged: /content/ml_engine_models_EUR_USD_20260106_174347.zip

📦 Contents:
Archive:  /content/ml_engine_models_EUR_USD_20260106_174347.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-01-06 17:39   trained_data/models/
        0  2026-01-06 17:39   trained_data/models/checkpoints/
     1900  2026-01-06 17:43   trained_data/models/modular_ensemble.meta.json
      910  2026-01-06 17:39   trained_data/models/training_report_20260105_233453.md
  1041488  2026-01-06 17:43   trained_data/models/xgb_momentum.pkl
     4355  2026-01-06 17:43   trained_data/models/transformer_direction.meta.pkl
     2201  2026-01-06 17:43   trained_data/models/ridge_confidence.pkl
      909  2026-01-06 17:39   trained_data/models/training_report_20260106_014935.md
   284478  2026-01-06 17:43   trained_data/models/transformer_direction.ema.pkl
   568792  2026-01-06 17:43   trained_data/models/transformer_direction.ewc.pkl
  1068828  2026-01-06 17:43   trained_da

In [84]:
# ============================================================
# 8.2 Download to local machine
# ============================================================
from google.colab import files

print("📥 Downloading models to your local machine...")
print("   (This will open a download dialog)\n")

files.download(f"/content/{zip_name}.zip")

print("\n✅ Download started!")
print("\n📋 To use on your Mac:")
print("   1. Unzip the downloaded file")
print("   2. Copy contents to ml_engine/trained_data/models/")
print("   3. Run: buddy analyze --model-type ensemble")

📥 Downloading models to your local machine...
   (This will open a download dialog)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started!

📋 To use on your Mac:
   1. Unzip the downloaded file
   2. Copy contents to ml_engine/trained_data/models/
   3. Run: buddy analyze --model-type ensemble


## 9️⃣ Optional: Push to GitHub

In [85]:
# ============================================================
# 9.1 Commit and push trained models to GitHub
# ============================================================
# ⚠️ Only run this if you want to push models to your repo

PUSH_TO_GITHUB = False  # Set to True to enable

if PUSH_TO_GITHUB:
    from getpass import getpass
    
    print("🔐 GitHub Authentication")
    print("Enter your GitHub Personal Access Token (PAT)")
    print("Create one at: https://github.com/settings/tokens\n")
    
    GITHUB_TOKEN = getpass("GitHub PAT: ")
    GITHUB_USER = input("GitHub Username: ")
    GITHUB_EMAIL = input("GitHub Email: ")
    
    # Configure git
    !git config --global user.name "{GITHUB_USER}"
    !git config --global user.email "{GITHUB_EMAIL}"
    
    # Set remote with token
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Raynergy-svg/ml_engine.git
    
    # Add and commit
    !git add trained_data/models/
    !git commit -m "feat: Add CUDA-trained models from Colab ({INSTRUMENT})"
    
    # Push
    !git push origin main
    
    print("\n✅ Models pushed to GitHub!")
else:
    print("ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.")

ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.


---

## 📝 Notes

### A100 Powerhouse Configuration
This notebook uses a **4x larger model** than Mac M1, optimized for A100's 80GB VRAM:

| Parameter | Mac M1 | A100 | Scale |
|-----------|--------|------|-------|
| d_model | 32 | 128 | 4x |
| num_heads | 4 | 8 | 2x |
| num_layers | 2 | 4 | 2x |
| dff | 64 | 512 | 8x |
| seq_len | 64 | 128 | 2x |
| batch_size | 64 | 256 | 4x |

### Optimal Data Size Calculator
The notebook automatically calculates optimal candle count based on model size:
- **Formula**: `optimal_candles = model_params × 50`
- **A100 Powerhouse**: ~60,000+ candles recommended (vs 12,000 previously)
- **Rule**: Larger models need more data to generalize well

### Multi-Pair Training
Buddy scan analyzes pairs for trading suitability:
- **Volatility**: Higher = more tradeable
- **Trend Strength**: Higher = more predictable
- **Clear Moves**: % of bars with direction > 0.05%
- **Score**: Weighted combination for training quality

Top 5 pairs by liquidity and scan score are used by default.

### GPU Memory Usage
- **T4 (16GB)**: Use batch_size=64, d_model=64, single pair
- **A100 (80GB)**: Full powerhouse config with multi-pair training

### Training Time Estimates (12K candles per pair × 5 pairs = 60K total)
- **T4 GPU**: ~45-60 minutes (reduced config, single pair)
- **A100 GPU**: ~15-25 minutes (full powerhouse, multi-pair)

### A100 Tensor Core Optimization
- All dimensions are multiples of 64 for maximum Tensor Core efficiency
- TF32 enabled for 1.5x speedup on matrix operations
- XLA compilation for fused kernel optimization

### Troubleshooting
- **OOM Error**: Reduce batch_size to 128 or d_model to 64
- **OANDA timeout**: Reduce CANDLES_PER_PAIR or fetch fewer pairs
- **Import errors**: Restart runtime and re-run setup cells
- **0% accuracy**: Make sure mixed precision is DISABLED (TF 2.19 bug)
- **"client" not defined**: Run cell 3.2 (Test OANDA connection) first